# 06 · PrimateAI — the semi-supervised alternative

**PrimateAI** (Sundaram et al. 2018, *Nat Genet*, PMID 30038395) is a deep net that dodges the need for curated pathogenic labels: it trains on a **proxy for benignity** — missense variants common in humans or seen in other primates. That makes it **semi-supervised**, with only **medium** circularity vs ClinVar.

> ✅ **REAL DATA.** PrimateAI's own native genome-wide release (Sundaram et al. 2018, Illumina BaseSpace, v0.2, hg38) for CFTR — **~9,722** variants (`data/primateai_cftr.csv`, built by a manual-download build cell below), scored under CFTR's canonical transcript. Near-saturating: CFTR's true missense universe is 9,730 (tools/05). **Coordinate-keyed** — join onto observed variants by `chrom,pos,ref,alt`. **Research use only** (Illumina). `source == 'REAL'`.


In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · PrimateAI — the *semi-supervised* alternative

**PrimateAI** (Sundaram et al. 2018, *Nature Genetics*, **PMID 30038395**) is a **deep neural
network**, but it dodges the "we need labelled pathogenic variants" problem with a neat trick.

Instead of training on curated *pathogenic* labels, it trains on a **proxy for benignity**:

> A missense variant that is **common** in healthy humans — or is seen as the normal amino acid in
> **other primates** (chimp, gorilla, …) — is very likely **tolerated / benign**. Evolution has
> already "tested" it.

So PrimateAI learns from **hundreds of thousands of common human & non-human-primate missense
variants** as its benign class. This is why it's called **semi-supervised**: it uses labels, but
those labels are a *population-frequency proxy*, **not** clinical pathogenic/benign assertions.

| Property | PrimateAI |
|---|---|
| Learning type | **Semi-supervised** (benign proxy from primate/common variants) |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Pathogenic cut | **≥ 0.803** |
| Circularity vs ClinVar | **Medium** — it never trained on ClinVar *pathogenic* labels, but common-variant proxies can still overlap benign ClinVar entries |


### Building the REAL data — a manual download (PrimateAI's own release)

dbNSFP also carries a PrimateAI_score column, but only for its ClinVar-associated
subset — covering just ~1,976 of CFTR's 9,730 possible missense variants (see the
coverage note below). This notebook instead uses **PrimateAI's own original
release** — the genome-wide scores Illumina published alongside Sundaram et al.
2018, distributed via BaseSpace. **You must fetch this one yourself:**

1. Go to the Illumina BaseSpace share <https://basespace.illumina.com/s/cPgCSmecvhb4>
   and download the release (PrimateAI scores for ~70M variants genome-wide).
2. Save the hg38 scores file as
   `data/primateAI/PrimateAI_scores_v0.2_hg38.tsv.gz` (gitignored — never commit it).

The cell below streams the ~910 MB gzip file **without loading it fully into
memory**, keeping only chromosome 7 rows inside the CFTR GRCh38 window. Unlike
REVEL's genome-wide table, this file is **not** chromosome-sorted (it's grouped
some other way — chromosomes appear out of order within the first few thousand
rows), so there's no early-exit optimization available; the full ~70M-row scan
takes well under a minute.

**Coverage: ~9,722 of CFTR's 9,730 true possible missense SNVs — effectively
saturating**, not the ~1,976 the dbNSFP subset gave. Every row is tagged with
exactly **one** UCSC transcript, `uc003vjd` — confirmed against UCSC's own gene
browser to be `ENST00000003084`, the same canonical CFTR transcript (MANE
`NM_000492.4`) every other tool in this toolkit already uses. No REVEL-style
multi-transcript disagreement to resolve here. Where this file and the old
dbNSFP subset overlap (1,976 variants), their scores are **byte-identical**
(verified: correlation 1.0, mean absolute difference 0.0) — dbNSFP was drawing
from this exact same release, just re-serving a much smaller slice of it.

> ⚠️ **Build gotcha:** the source file uses UCSC-style chromosome names (`chr7`),
> while every other tool in this toolkit (gnomAD, REVEL, CFTR2, ...) uses the bare
> Ensembl-style form (`7`) for the coordinate join. The build cell strips the
> `chr` prefix — miss this and PrimateAI would silently join zero rows against
> anything else in the toolkit, no error raised.

**Coordinate-keyed, no protein position.** The source file gives `refAA`/`altAA`
but not a residue number, so there's no `protein_variant` column here — join
by `chrom,pos,ref,alt` (as `predict/13_cftr2_benchmark.ipynb` already does), or
bridge to a protein name via gnomAD's `protein_variant` when you need one (see
the worked-example panel below for the pattern — the same one REVEL uses,
since REVEL's source file has the same limitation).

**License: research use only.** The release file's own header states, verbatim:
*"Copyright 2018 Illumina Inc... For research use only."* — a direct restriction
from Illumina, not dbNSFP's CC BY-NC-ND terms (dbNSFP isn't the source here). See
`data_manifest.json`.

**Version & reproducibility.** Unlike the dbNSFP parquet (no useful embedded
metadata at all), this gzip file carries a real timestamp in its own format
header — gzip's `MTIME` field, readable without decompressing anything:
**2018-09-04 21:19:43 UTC**, matching the file's actual publication shortly
after the PrimateAI paper. The build cell reads it directly and records it into
`data/primateai_cftr.release.json`; `load_primateai()` exposes it as the
`primateai_release` column.


In [2]:
import gzip, csv, json, struct
from datetime import datetime, timezone

DATA_DIR = pathlib.Path.cwd().parent / "data"
PRIMATEAI_GZ = DATA_DIR / "primateAI" / "PrimateAI_scores_v0.2_hg38.tsv.gz"
PRIMATEAI_TSV = DATA_DIR / "primateai_cftr.csv"
PRIMATEAI_RELEASE_JSON = DATA_DIR / "primateai_cftr.release.json"
CFTR_WINDOW = (117_470_000, 117_670_000)   # GRCh38


def gzip_mtime(path: pathlib.Path) -> str:
    """Read the MTIME field straight out of the gzip format header -- no decompression needed."""
    with open(path, "rb") as fh:
        header = fh.read(10)
    _magic1, _magic2, _method, _flags, mtime, _extra, _os = struct.unpack("<BBBBIBB", header)
    return datetime.fromtimestamp(mtime, tz=timezone.utc).isoformat() if mtime else "unknown"


if PRIMATEAI_TSV.exists():
    print(f"already built -> {PRIMATEAI_TSV.name} (delete it and {PRIMATEAI_RELEASE_JSON.name} to rebuild)")
elif not PRIMATEAI_GZ.exists():
    raise FileNotFoundError(
        f"{PRIMATEAI_GZ} not found.\n"
        "PrimateAI's native release has no small per-gene download -- get the full\n"
        "genome-wide hg38 scores file:\n"
        "  1. Go to https://basespace.illumina.com/s/cPgCSmecvhb4 and download the\n"
        "     PrimateAI v0.2 hg38 scores (~910 MB gzipped, ~70M variants)\n"
        f"  2. Save it as {PRIMATEAI_GZ} (do NOT commit it -- data/ is gitignored)\n"
        "Then re-run this cell -- it streams the file and keeps only the CFTR window."
    )
else:
    release_date = gzip_mtime(PRIMATEAI_GZ)
    print(f"release file packaged at (gzip-embedded timestamp): {release_date}")

    start, end = CFTR_WINDOW
    rows, n = [], 0
    with gzip.open(PRIMATEAI_GZ, "rt") as fh:
        header = None
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            if header is None:
                header = line.rstrip("\n").split("\t")
                idx = {c: i for i, c in enumerate(header)}
                continue
            n += 1
            if line.startswith("chr7\t"):
                f = line.rstrip("\n").split("\t")
                p = int(f[idx["pos"]])
                if start <= p <= end:
                    chrom = f[idx["chr"]].removeprefix("chr")  # normalize to bare "7", matching gnomAD/REVEL/CFTR2
                    rows.append((chrom, p, f[idx["ref"]], f[idx["alt"]],
                                 f[idx["refAA"]], f[idx["altAA"]],
                                 float(f[idx["primateDL_score"]]), float(f[idx["ExAC_coverage"]]),
                                 f[idx["UCSC_gene"]]))
    print(f"scanned {n:,} PrimateAI rows genome-wide (file is not chromosome-sorted, so no early exit)")
    df = pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "aaref", "aaalt",
                                      "primate_ai_score", "exac_coverage", "ucsc_transcript"])
    df = df.drop_duplicates(["chrom", "pos", "ref", "alt"]).sort_values("pos").reset_index(drop=True)
    df["source"] = "REAL"
    df.to_csv(PRIMATEAI_TSV, index=False)
    PRIMATEAI_RELEASE_JSON.write_text(json.dumps({
        "source_file": PRIMATEAI_GZ.name,
        "gzip_embedded_date": release_date,
        "note": "gzip_embedded_date is read from the file's own gzip MTIME header field, "
                "not a download date -- Illumina's real record of when this file was packaged.",
    }, indent=2))
    print(f"REAL PrimateAI CFTR variants written: {len(df):,} -> {PRIMATEAI_TSV.relative_to(DATA_DIR.parent)}")
    print(f"wrote {PRIMATEAI_RELEASE_JSON.relative_to(DATA_DIR.parent)}")
    print(f"unique UCSC transcript(s) seen: {df['ucsc_transcript'].unique().tolist()}")


release file packaged at (gzip-embedded timestamp): 2018-09-04T21:19:43+00:00


scanned 70,116,384 PrimateAI rows genome-wide (file is not chromosome-sorted, so no early exit)


REAL PrimateAI CFTR variants written: 9,722 -> data\primateai_cftr.csv
wrote data\primateai_cftr.release.json
unique UCSC transcript(s) seen: ['uc003vjd.3']


In [3]:
tk.THRESHOLDS['primate_ai']

{'path': 0.803, 'benign': 0.483}

## 2 · Load PrimateAI and make a call

Score 0-1, higher = worse, pathogenic cut `>= 0.803`.

In [4]:
primateai = tk.load_primateai()   # REAL — PrimateAI's native release for CFTR (~9,722), built above
print(f"{len(primateai):,} REAL PrimateAI variants | source: {primateai['source'].unique().tolist()}")
print('columns:', list(primateai.columns))
print('score range:', primateai['primate_ai_score'].min(), '->', primateai['primate_ai_score'].max(),
      '| pathogenic (>= 0.803):', int((primateai['primate_ai_score'] >= 0.803).sum()))
print('primateai_release (gzip-embedded timestamp):', primateai['primateai_release'].iloc[0])
primateai['pai_call'] = primateai['primate_ai_score'].apply(lambda s: tk.call_from_score(s, 'primate_ai'))
primateai[['chrom', 'pos', 'ref', 'alt', 'primate_ai_score', 'pai_call', 'source']].head(10)


9,722 REAL PrimateAI variants | source: ['REAL']
columns: ['chrom', 'pos', 'ref', 'alt', 'aaref', 'aaalt', 'primate_ai_score', 'exac_coverage', 'ucsc_transcript', 'source', 'primateai_release']
score range: 0.176448047161 -> 0.932814955711 | pathogenic (>= 0.803): 521
primateai_release (gzip-embedded timestamp): 2018-09-04T21:19:43+00:00


,chrom,pos,ref,alt,primate_ai_score,pai_call,source
0,7,117480098,C,A,0.646725,uncertain,REAL
1,7,117480098,C,G,0.613581,uncertain,REAL
2,7,117480099,A,C,0.612750,uncertain,REAL
3,7,117480099,A,G,0.640190,uncertain,REAL
4,7,117480099,A,T,0.603858,uncertain,REAL
5,7,117480100,G,C,0.640343,uncertain,REAL
6,7,117480100,G,T,0.640343,uncertain,REAL
7,7,117480101,A,G,0.384172,benign,REAL
8,7,117480101,A,T,0.439323,benign,REAL
9,7,117480102,G,T,0.399057,benign,REAL


### A note on keying

PrimateAI's native file is genomic-coordinate-keyed with no protein-position
column (see "Building the REAL data" above for why, and how the worked-example
panel below bridges to a protein name via gnomAD when it needs one). CFTR is on
the **plus** strand, so the coding alleles and the genomic `ref`/`alt` are the
same — no complementing needed.


In [5]:
info = tk.TOOL_REGISTRY['PrimateAI']
for key, val in info.items():
    print(f'  {key:12s}: {val}')

  kind        : missense
  learning    : semi-supervised
  signal      : deep net trained on common human/primate missense as a benign proxy
  circularity : medium
  pmid        : 30038395


## Example: the shared missense worked-example panel, scored by **PrimateAI**

The same fixed panel of famous CFTR **missense** variants runs through every missense tool
(tools/01–06, benchmark/00–01), so you can follow one set of variants across the series. The
variant list is `tk.A1_PANEL_VARIANTS` / `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the
**scoring is shown inline below** so you can see exactly how PrimateAI is joined onto it.

In [6]:
# PrimateAI's native file is COORDINATE-keyed (no protein position). To score the
# panel we bridge each protein key -> genomic coordinate via gnomAD, then look
# PrimateAI up by coordinate — the same join-key lesson REVEL's panel cell shows.
panel = tk.A1_PANEL_VARIANTS
coord = tk.load_gnomad_missense().drop_duplicates('protein_variant').set_index('protein_variant')['variant_id']
pai = tk.load_primateai()
rows = []
for pv in panel:
    vid = coord.get(pv)
    score = None
    if isinstance(vid, str) and vid.count('-') == 3:
        _c, p, ref, alt = vid.split('-')
        hit = pai[(pai['pos'] == int(p)) & (pai['ref'] == ref) & (pai['alt'] == alt)]
        score = round(float(hit['primate_ai_score'].iloc[0]), 4) if len(hit) else None
    rows.append({'protein_variant': pv, 'gnomad_coord': vid, 'primate_ai_score': score})
pd.DataFrame(rows)


,protein_variant,gnomad_coord,primate_ai_score
0,G551D,7-117587806-G-A,0.7570
1,F508del,None,NaN
2,R117H,7-117530975-G-A,0.5386
3,R334W,7-117540230-C-T,0.7153
4,G85E,7-117509123-G-A,0.7759
5,D1152H,7-117614699-G-C,0.6876
6,R668C,7-117592169-C-T,0.7138
7,Y161C,7-117531107-A-G,0.8441
8,G970D,7-117606674-G-A,0.6212
9,S912L,7-117603609-C-T,0.3013


## Key takeaways

1. **PrimateAI** is **semi-supervised** (benignity learned from common human/primate variants) → **medium** circularity. Cut `>= 0.803`.
2. Now **REAL** — PrimateAI's own native genome-wide release (Sundaram et al. 2018, Illumina
   BaseSpace v0.2, hg38). Coverage: **~9,722** of CFTR's 9,730 true possible missense SNVs
   (tools/05) — effectively saturating (dbNSFP's PrimateAI_score column, by contrast, only covers
   its ClinVar-associated subset, ~1,976 for CFTR). Verified: every row scored under one
   confirmed-canonical UCSC transcript (`uc003vjd` = `ENST00000003084`), and scores agree exactly
   with dbNSFP's values wherever the two overlap — same underlying release, different slice.
3. **Coordinate-keyed, not protein-keyed** — no protein-position column in the source; bridge to
   `protein_variant` via gnomAD when needed (see the worked-example panel above).
4. **Version-tracked via the file format itself**: gzip's own `MTIME` header field gives a real
   timestamp (`primateai_release` = 2018-09-04T21:19:43+00:00) with no parsing or downloading needed.
5. **Research use only** (Illumina, 2018) — more specific than a generic non-commercial note; raw
   release file kept external.

